# 5.2 — MuscleMimic Fullbody: Train a Policy (CPU) and Ghost-Body Visualization

End-to-end walkthrough of CPU-based MuscleMimic training and ghost-body rendering:

1. Locate a retargeted motion clip
2. Create `MuscleMimicClipEnvV0` — the Gymnasium environment
3. Train an ActorCritic policy with PPO (`train_mimic.py`)
4. Save and reload a checkpoint
5. Build a `GhostBodyViz` to overlay reference motion
6. Render an inline MP4 video
7. Use the CLI render script for interactive or MP4 export

**NOTE:** The motion clips come from the gated Hugging Face dataset https://huggingface.co/datasets/amathislab/musclemimic-retargeted. You may need to first open that page and accept the licence requirements; afterwards, create a Hugging Face access token and make it available, e.g. via `export HF_TOKEN=...` before starting Jupyter, or `hf auth login`.

## 5.2.0 — Paths

In [ ]:
import sys
from pathlib import Path

# Resolve the repo root regardless of where the kernel CWD lands
_NOTEBOOK_DIR = Path(__file__).parent if "__file__" in dir() else Path.cwd()
_REPO_ROOT = _NOTEBOOK_DIR
while not (_REPO_ROOT / "myosuite").is_dir():
    parent = _REPO_ROOT.parent
    if parent == _REPO_ROOT:
        raise RuntimeError("Cannot find myosuite repo root")
    _REPO_ROOT = parent

if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))
if str(_REPO_ROOT / "tutorials" / "files/5.2") not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT / "tutorials" / "files/5.2"))

print("Repo root :", _REPO_ROOT)


In [ ]:
CLIP_PATH = None
try:
    from pathlib import Path
    from huggingface_hub import hf_hub_download

    CLIP_REPO_ID = 'amathislab/musclemimic-retargeted'
    CLIP_FILENAME = 'MyoFullBody/gmr/KIT/167/walking_medium06_poses.npz'

    CLIP_PATH = Path(
        hf_hub_download(
            repo_id=CLIP_REPO_ID,
            filename=CLIP_FILENAME,
            repo_type='dataset',
        )
    )
    print('Clip:', CLIP_PATH)
except Exception as e:
    print(f'[SKIP] clip download failed: {e}')
    CLIP_PATH = None


## 5.2.1 — Locate the motion clip

In [ ]:
CLIP_PATH = None
try:
    import numpy as np
    from pathlib import Path
    from huggingface_hub import hf_hub_download

    CLIP_REPO_ID = 'amathislab/musclemimic-retargeted'
    CLIP_FILENAME = 'MyoFullBody/gmr/KIT/167/walking_medium06_poses.npz'

    CLIP_PATH = Path(
        hf_hub_download(
            repo_id=CLIP_REPO_ID,
            filename=CLIP_FILENAME,
            repo_type='dataset',
        )
    )

    if not CLIP_PATH.is_file():
        raise FileNotFoundError(
            f'Clip not found after hf_hub_download: {CLIP_PATH}\n'
            'Check internet/auth and verify CLIP_REPO_ID/CLIP_FILENAME.'
        )

    npz = np.load(CLIP_PATH, allow_pickle=True)
    print('Clip keys      :', list(npz.keys()))
    print('qpos shape     :', npz['qpos'].shape)
    print('qvel shape     :', npz['qvel'].shape)
    print('site_xpos shape:', npz['site_xpos'].shape)
    print('xpos shape     :', npz['xpos'].shape)
    print('xquat shape    :', npz['xquat'].shape)
except Exception as e:
    print(f'[SKIP] clip download failed: {e}')
    CLIP_PATH = None


## 5.2.2 — Create the simulation environment

`MuscleMimicClipEnvV0` wraps the MyoSuite full-body model with:

| Feature | Details |
|---------|---------|
| Obs | `qpos(89) + qvel(88) + act(354) + lookahead(290)` = 821 |
| Act | 354 muscle excitations ∈ [0, 1] |
| Reward | DeepMimic composite (site tracking 0.6, joint pos/vel 0.1, root kinematics 0.2) |
| Termination | Site error > 1 m **or** root error > 0.3 m |
| Reset | Random start frame in clip |


In [ ]:
if CLIP_PATH is None:
    print('[SKIP] no motion clip')
else:
    from myosuite.envs.myo.tasks.mimic.clip_env import MuscleMimicClipEnvV0

    env = MuscleMimicClipEnvV0(clip_path=CLIP_PATH, max_episode_steps=500, random_start=True)

    obs_dim = env.observation_space.shape[0]
    act_dim = env.action_space.shape[0]

    obs, info = env.reset()
    print(f"obs shape  : {obs.shape}   (obs_dim={obs_dim})")
    print(f"act_dim    : {act_dim}")
    print(f"clip length: {env._clip_T} frames")
    print(f"ctrl_dt    : {env._ctrl_dt:.4f} s")

    # One random step to verify reward computation
    action = env.action_space.sample()
    obs2, rew, term, trunc, info2 = env.step(action)
    print(f"\nstep reward  : {rew:.4f}  terminated={term}  truncated={trunc}")
    print(f"site tracking: {info2['site']:.4f}")
    print(f"joint pos    : {info2['joint_pos']:.4f}")
    print(f"root orient  : {info2['root_orient']:.4f}")


## 5.2.3 — PPO training

`VecMimicEnv` runs N synchronous environments in-process.
`ActorCritic` is a configurable-depth MLP with SiLU+LayerNorm residual blocks.

The cell below trains for **50 k control steps** (~30 s on CPU) as a demo.
For real convergence use `total_steps=2_000_000_000` with a GPU.


In [ ]:
policy = None
if CLIP_PATH is None:
    print('[SKIP] no motion clip; skipping PPO demo')
else:
    import torch
    from train_mimic import ActorCritic, VecMimicEnv, train

    torch.manual_seed(42)

    import os
    _FULL = os.environ.get('MYOSUITE_FULL_MIMIC', '0') == '1'
    DEMO_STEPS    = 50_000 if _FULL else 256  # 50k is a longer demo; 2e9 for paper-scale
    N_ENVS        = 4 if _FULL else 1
    ROLLOUT_STEPS = 64

    print(f"Training for {DEMO_STEPS:,} steps  ({N_ENVS} envs × {ROLLOUT_STEPS} rollout) ...")

    policy = train(
        clip_path     = CLIP_PATH,
        total_steps   = DEMO_STEPS,
        n_envs        = N_ENVS,
        rollout_steps = ROLLOUT_STEPS,
        hidden_dim    = 256,
        n_layers      = 4,
        lr            = 3e-4,
        device        = "cpu",  #"cuda",
        log_interval  = 5,
    )

    print(f"\nTraining complete.  Policy parameters: {sum(p.numel() for p in policy.parameters()):,}")


## 5.2.4 — Save and reload a checkpoint

Pass `save_path` to `train()` to write the checkpoint, or save manually.


In [ ]:
if policy is None:
    print('[SKIP] no policy')
else:
    CKPT_PATH = Path("/tmp/mimic_demo_policy.pt")
    torch.save(policy.state_dict(), CKPT_PATH)
    print(f"Checkpoint saved → {CKPT_PATH}")

    # Reload ----------------------------------------------------------------
    _vec = VecMimicEnv(clip_path=CLIP_PATH, n_envs=1, max_episode_steps=50)
    loaded_policy = ActorCritic(_vec.obs_dim, _vec.act_dim, hidden_dim=256, n_layers=4)
    loaded_policy.load_state_dict(torch.load(CKPT_PATH, weights_only=True, map_location="cpu"))
    loaded_policy.eval()
    print(f"Checkpoint reloaded — {sum(p.numel() for p in loaded_policy.parameters()):,} params")


In [ ]:
# from myosuite.integrations.musclemimic.fullbody_local_policy import (
#     load_local_policy_artifacts,
# )
# from myosuite.integrations.musclemimic.actor_torch import MimicActorModule
# from myosuite.integrations.musclemimic.actor_onnx import (
#     export_to_onnx,
#     load_onnx_session,
# )

# # artifacts = load_local_policy_artifacts(CKPT_PATH)
# # module = MimicActorModule.from_artifacts(artifacts)

# # onnx_path = export_to_onnx(policy, Path("/tmp/mimic_demo_policy.onnx"))
# session = load_onnx_session("/tmp/mimic_demo_policy.pt")

# # Batched inference: (N, obs_dim) → (N, act_dim)
# obs_np = np.zeros((N, artifacts.obs_dim), dtype=np.float32)
# actions = session.act(obs_np)

## 5.2.5 — Ghost-body visualization

`GhostBodyViz` reads the precomputed `xpos / xquat` body-position arrays from
the clip and draws a semi-transparent skeleton into `viewer.user_scn` (or
`renderer.scene`) each frame using `mjv_initGeom` — **no second physics
simulation** is needed.

Geometry drawn per visible body:
- **Capsule** connecting parent → child (semi-transparent blue)
- **Sphere** at the body centre (lighter blue)


In [ ]:
if CLIP_PATH is None:
    print('[SKIP] no motion clip')
else:
    import mujoco
    import numpy as np
    from myosuite.viz.ghost_body_viz import GhostBodyViz

    # In replay mode the ghost xpos matches the physics body exactly —
    # they'd be completely overlapping and invisible.  A lateral offset
    # places the ghost 1.5 m to the side for a clear side-by-side view.
    # When running a policy (USE_POLICY=True below) the bodies diverge
    # naturally so you can set position_offset=None.
    GHOST_OFFSET = None  #np.array([0.0, 1.5, 0.0])   # 1.5 m along Y axis

    ghost_viz = GhostBodyViz.from_clip_and_model(
        CLIP_PATH,
        env.model,
        skeleton_rgba    = (0.3, 0.6, 1.0, 0.6),   # blue, 60 % opacity
        joint_rgba       = (0.5, 0.8, 1.0, 0.8),   # lighter blue, 80 % opacity
        capsule_radius   = 0.022,
        joint_radius     = 0.028,
        position_offset  = GHOST_OFFSET,
    )

    print(f"Clip frames    : {ghost_viz._T}")
    print(f"Visible bodies : {len(ghost_viz.visible_body_ids)}")
    print(f"Position offset: {ghost_viz.position_offset}")

    # Verify draw() appends geoms without crashing
    _scene = mujoco.MjvScene(env.model, maxgeom=4096)
    _scene.ngeom = 0
    ghost_viz.draw(0, _scene)
    print(f"Geoms after draw(frame=0): {_scene.ngeom}")


## 5.2.6 — Render an inline video

Renders N frames using `mujoco.Renderer`:
- **Physics body** in neutral pose (or policy-driven when `USE_POLICY=True`)
- **Ghost skeleton** overlaid from the reference clip

The ghost is written into `renderer.scene` immediately after `update_scene(data)`.


In [ ]:
if CLIP_PATH is None or policy is None:
    print('[SKIP] clip/policy missing')
else:
    from IPython.display import Video, display

    # ── parameters ────────────────────────────────────────────────────────────
    N_RENDER_FRAMES = 200
    RENDER_W, RENDER_H = 640, 360
    VIDEO_OUT  = Path("/tmp/mimic_ghost_render.mp4")
    # False → replay reference clip (ghost offset makes ghost visible beside body)
    # True  → run loaded_policy (bodies diverge, set GHOST_OFFSET=None for overlay)
    USE_POLICY = True

    # ── renderer ──────────────────────────────────────────────────────────────
    renderer = mujoco.Renderer(env.model, height=RENDER_H, width=RENDER_W)
    ghost_fn  = ghost_viz.as_viz_fn()

    scene_opt = mujoco.MjvOption()
    scene_opt.flags[mujoco.mjtVisFlag.mjVIS_ACTIVATION] = 1
    scene_opt.flags[mujoco.mjtVisFlag.mjVIS_ACTUATOR] = 1
    # optional: scene_opt.flags[mujoco.mjtVisFlag.mjVIS_TENDON] = 1

    frames = []
    obs, _ = env.reset()
    # Start from first clip frame for reproducible output
    env.data.qpos[:] = env._clip_qpos[0]
    env.data.qvel[:] = env._clip_qvel[0]
    mujoco.mj_forward(env.model, env.data)

    for step_idx in range(N_RENDER_FRAMES):
        if USE_POLICY:
            obs_t = torch.as_tensor(obs, dtype=torch.float32).unsqueeze(0)
            with torch.no_grad():
                action, _, _, _ = loaded_policy.get_action_and_value(obs_t)
            obs, _, term, trunc, _ = env.step(action.squeeze(0).numpy())
            if term or trunc:
                obs, _ = env.reset()
        else:
            frame = step_idx % env._clip_T
            env.data.qpos[:] = env._clip_qpos[frame]
            env.data.qvel[:] = env._clip_qvel[frame]
            mujoco.mj_forward(env.model, env.data)

        renderer.update_scene(env.data, scene_option=scene_opt)
        ghost_fn(step_idx, env.model, env.data, renderer.scene)
        frames.append(renderer.render().copy())

    renderer.close()

    # Quick sanity check: count blue ghost pixels in first frame
    _f = frames[0]
    _blue = (_f[:,:,2] > 150) & (_f[:,:,0] < 120)
    print(f"Rendered {len(frames)} frames at {RENDER_W}x{RENDER_H}")
    print(f"Ghost blue pixels in frame 0: {_blue.sum()} (should be > 0)")

    # ── write video ────────────────────────────────────────────────────────────
    try:
        import mediapy
        mediapy.write_video(str(VIDEO_OUT), frames, fps=30)
        print(f"Saved → {VIDEO_OUT}")
        display(Video(str(VIDEO_OUT), embed=True, html_attributes="controls loop"))
    except ImportError:
        import os, imageio
        os.makedirs("/tmp/mimic_frames", exist_ok=True)
        for i, f in enumerate(frames):
            imageio.imwrite(f"/tmp/mimic_frames/frame_{i:04d}.png", f)
        print(f"mediapy not installed — saved {len(frames)} PNGs to /tmp/mimic_frames/")
        print("Install with: pip install mediapy")


## 5.2.7 — CLI render script

For interactive viewing (requires a display) or long MP4 exports, use
`render_mimic.py` directly. You can resolve the clip path portably with
`hf_hub_download(...)`:

```bash
CLIP_PATH=$(python -c "from huggingface_hub import hf_hub_download; print(hf_hub_download(repo_id='amathislab/musclemimic-retargeted', filename='MyoFullBody/gmr/KIT/167/walking_medium06_poses.npz', repo_type='dataset'))")

# Interactive passive viewer — ghost overlay + reference replay
python tutorials/files/5.2/render_mimic.py \
    --clip "$CLIP_PATH"

# Interactive viewer with a trained policy
python tutorials/files/5.2/render_mimic.py \
    --clip "$CLIP_PATH" \
    --checkpoint /tmp/mimic_demo_policy.pt

# Export 500 frames to MP4
python tutorials/files/5.2/render_mimic.py \
    --clip "$CLIP_PATH" \
    --checkpoint /tmp/mimic_demo_policy.pt \
    --output render.mp4 --n-frames 500 --fps 30
```

| Flag | Default | Meaning |
|------|---------|---------|
| `--clip` | required | Path to `.npz` motion clip |
| `--checkpoint` | None | ActorCritic `.pt` state dict; omit for replay |
| `--output` | None | Write MP4 here instead of opening viewer |
| `--n-frames` | 500 | Frames for `--output` mode |
| `--playback-speed` | 1.0 | Real-time speed multiplier (viewer only) |
| `--skeleton-alpha` | 0.35 | Ghost skeleton opacity |


## 5.2.8 — Architecture overview

```
motion clip (.npz)
  ├── qpos / qvel  (T, nq/nv)     → env state initialisation + DeepMimic reward
  ├── site_xpos   (T, 17, 3)      → site tracking term (w=0.6)
  └── xpos / xquat (T, 102, 3/4)  → GhostBodyViz capsule positions

MuscleMimicClipEnvV0 (gymnasium.Env)
  ├── obs  = qpos(89) + qvel(88) + act(354) + lookahead(290) = 821
  ├── rew  = mimic_composite_reward(xp=np, ...)  ← 6-term DeepMimic
  └── done = mimic_should_terminate(site_err > 1 m | root_err > 0.3 m)

ActorCritic (MLP, SiLU+LayerNorm)
  ← VecMimicEnv  (N synchronous environments)
  ← train()      → PPO with GAE(λ=0.95), clip ε=0.2, linear LR anneal
  → checkpoint .pt

GhostBodyViz
  from_clip_and_model(clip_path, model)
    └── draw(step_idx, user_scn)  ← mjv_initGeom capsules + spheres
  as_viz_fn() → (step_idx, model, data, scene) → None
```

### Key files

| File | Purpose |
|------|---------|
| `myosuite/envs/myo/tasks/mimic/clip_env.py` | CPU Gymnasium env |
| `myosuite/terms/mimic_reward.py` | Backend-agnostic DeepMimic reward |
| `myosuite/terms/mimic_obs.py` | k-step lookahead obs builder |
| `myosuite/physics/running_stats.py` | Welford online normaliser |
| `myosuite/viz/ghost_body_viz.py` | Semi-transparent ghost skeleton |
| `tutorials/files/5.2/train_mimic.py` | Self-contained PPO training |
| `tutorials/files/5.2/render_mimic.py` | CLI render / MP4 export |
